In [1]:
from langchain_community.document_loaders import PyPDFLoader
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [2]:



file_path = r"/Users/rutvik/Developer/github/Medical-AI-Chatbot/data/Gale Encyclopedia of Medicine. Vol. 1. 2nd Edition ( PDFDrive ).pdf"
loader = PyPDFLoader(file_path)
document = loader.load()

chunk_size=500
chunk_overlap=20
text_splitter = RecursiveCharacterTextSplitter(
chunk_size=chunk_size,
chunk_overlap=chunk_overlap,
length_function=len,
is_separator_regex=False,
)

# Split the document into chunks
splitted_doc = text_splitter.split_documents(document)
splitted_doc = splitted_doc[:100]

In [5]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
embedding_model="hkunlp/instructor-xl"
search_type="similarity"
k=5

# Initialize HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# Chroma DB path
persist_directory = "data"
collection_name = "chroma_collection"

# Load or create the Chroma vector store
chroma_db = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings,
    collection_name=collection_name
)

# Check if the Chroma database has any existing vectors
if not chroma_db._collection or not chroma_db._collection.count():
    print("Chroma database is empty. Creating a new one...")

    # Create and persist the Chroma database
    chroma_db = Chroma.from_documents(
        documents=splitted_doc,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    print("Chroma is in persist mode.")
    chroma_db.persist()

# Set up the retriever with specified search type and parameters
retriever = chroma_db.as_retriever(search_type=search_type, search_kwargs={"k": k})


/var/folders/5t/ftdvh18x6xv506mg6dwy_mzm0000gn/T/ipykernel_4602/355974546.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
/Users/rutvik/Developer/github/Medical-AI-Chatbot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/rutvik/Developer/github/Medical-AI-Chatbot/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be r

ChunkedEncodingError: ('Connection broken: IncompleteRead(195429469 bytes read, 982916190 more expected)', IncompleteRead(195429469 bytes read, 982916190 more expected))

In [4]:
!pip install langchain_community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 6.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

context_placeholder="{context}"
input_placeholder="{input}"

system_prompt = f"""
    You are a specialized AI assistant designed for medical question-answering.

    **User Greeting Behavior:**
    - If the user input is a greeting (e.g., "hey", "hello", "hi"), respond warmly with **varied friendly replies**, such as:
        - "Hey! How can I assist you today?"
        - "Hello! Hope you're doing well. How can I help?"
        - "Hi there! What medical question do you have?"
    - Do **NOT** retrieve medical context for greetings.

    **Answer Formatting Guidelines:**
    - Format your answers **clearly and concisely**.
    - Use **bulleted lists** for multiple points.
    - Provide step-by-step information where applicable.
    - If listing symptoms, treatments, or precautions, use the following format:
        **Example Format:**
        - **Symptoms:** Fever, fatigue, shortness of breath.
        - **Treatment Options:**
            - Medication: [Specify]
            - Lifestyle Changes: [Specify]
        - **When to See a Doctor:** If symptoms persist for more than X days.

    **Strict Knowledge Boundaries:**
    - Answer **ONLY** using the retrieved medical context provided to you.
    - DO NOT use any outside knowledge or make assumptions.
    - If the retrieved context does not contain enough information to answer, respond with:
      *"I’m sorry, but I couldn't find relevant information in the available data. Please consult a medical professional for accurate advice."*

    **Medical Context:**
    {context_placeholder}
    """

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", input_placeholder)
    ]
)


In [69]:
from langchain_ollama.chat_models import ChatOllama

llm = ChatOllama(
model='deepseek-r1:1.5b',
temperature=0.4
)


In [83]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import RetrievalQA
import logging
import re

query = "what is dialysis? and types of dialysis?"
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

answer = rag_chain.invoke({"input": query})
translated_text = re.sub(r'<think>.*?</think>\n\n', '', answer['answer'], flags=re.DOTALL)


In [84]:
answer['answer']

"<think>\nOkay, so I need to figure out what dialysis is and its different types based on the medical context provided. Let me start by reading through the given information again.\n\nThe context mentions that dialysis involves circulating blood outside the body using a dialyzer. It says there are three types of peritoneal dialysis, but it doesn't specify if there are other types. The main part I remember is about hemodialysis and hemofiltration.\n\nWait, the user asked specifically for dialysis types. The context didn't mention anything beyond peritoneal dialysis. So maybe dialysis isn't just peritoneal? Or perhaps dialysis includes both peritoneal and other methods?\n\nI know there are two main forms of dialysis: hemodialysis (HD) and peritoneal dialysis (PD). HD uses a dialyzer in the blood, while PD is done through a catheter in the abdomen. Both are used for managing kidney function.\n\nBut looking back at the context, it only mentions peritoneal dialysis. So maybe dialysis refers

In [85]:

print(translated_text)

Dialysis encompasses two primary forms: **hemodialysis (HD)** and **peritoneal dialysis (PD)**. Hemodialysis involves circulating blood outside the body using a dialyzer in the bloodstream, typically performed in a hospital setting with a dialyzer machine. Peritoneal dialysis uses a flexible catheter inserted into the abdomen to filter fluids from the blood, often used for patients with acute kidney failure or those requiring fluid management. Both methods are essential for managing kidney function and improving patient outcomes.
